# 美联储货币政策与全球资产价格传导 — 数据获取模块

| 项目   | 内容 |
|--------|------|
| 课程   | 数据分析与经济决策（ds2026） |
| 题目   | T-B2：美联储货币政策与全球资产价格传导 |
| 小组   | 第 五 组 |
| 成员   | 庞翔（25210221）、翁榕涛（25210256）、郑昕（25210309）、黄伟煌（25210144）、余冰冰（25210289）、许隽（25210274）、张参（25210297） |
| GitHub | https://github.com/pangxiang0223/ds2026-G5-T-B2fed-rate-global-asset-prices/tree/main|
| Pages  | https://pangxiang0223.github.io/ds2026-G5-T-B2fed-rate-global-asset-prices/ |
| 日期   | 2025-05-14 |

## 任务说明
本步骤目标：使用 fredapi 获取美联储宏观经济数据（利率、通胀、货币供应量）、使用 yfinance 获取股指、黄金、债券 ETF 的历史价格，为后续数据清洗、周期分析、可视化提供完整、无缺失的原始数据。

In [9]:
## code
from fredapi import Fred
import pandas as pd
from config import FRED_API_KEY  # 见 T-A3 说明

fred = Fred(api_key=FRED_API_KEY)

# 联邦基金利率目标（有效利率，日度）
fedfunds = fred.get_series('FEDFUNDS', observation_start='1993-01-01')

# M2 货币供应量（月度，十亿美元）
m2 = fred.get_series('M2SL', observation_start='1993-01-01')

# PCE 通胀（月度，同比%）
pce = fred.get_series('PCEPI', observation_start='1993-01-01')
pce_yoy = pce.pct_change(12) * 100  # 转为同比增速

# 整理为 DataFrame
macro = pd.DataFrame({
    'fedfunds': fedfunds,
    'm2': m2,
    'pce_yoy': pce_yoy,
}).resample('ME').last()

macro.to_csv('data_raw/fed_rates_raw.csv')
print(macro.tail())

            fedfunds       m2   pce_yoy
2025-12-31      3.72  22353.6  2.878084
2026-01-31      3.64  22429.3  2.856869
2026-02-28      3.64  22627.3  2.829552
2026-03-31      3.64  22686.0  3.496081
2026-04-30      3.64      NaN       NaN


In [7]:
%pip install fredapi yfinance pandas matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
## code
import yfinance as yf

# 各类资产代码说明
assets = {
    '^GSPC':  'SP500',       # 标普 500 指数
    '^NDX':   'Nasdaq100',   # 纳斯达克 100
    'GLD':    'Gold_ETF',    # 黄金 ETF（SPDR）
    'TLT':    'TBond_20Y',   # 20 年期美债 ETF
    'BTC-USD':'Bitcoin',     # 比特币（2014 年起有数据）
    'DX-Y.NYB':'DXY',        # 美元指数
    '^VIX':   'VIX',         # 恐慌指数
}

prices = yf.download(
    list(assets.keys()),
    start='1993-01-01',
    auto_adjust=True
)['Close']

prices.columns = [assets[c] for c in prices.columns]
prices_monthly = prices.resample('ME').last()
prices_monthly.to_csv('data_raw/assets_raw.csv')
print(prices_monthly.tail())

[*********************100%***********************]  7 of 7 completed

                 Bitcoin        DXY    Gold_ETF  TBond_20Y        SP500  \
Date                                                                      
2026-01-31  78621.117188  96.989998  444.950012  85.849037  6939.029785   
2026-02-28  66995.859375  97.610001  483.750000  89.827065  6878.879883   
2026-03-31  68233.312500  99.959999  430.290009  86.027336  6528.520020   
2026-04-30  76304.320312  98.080002  423.660004  85.305000  7209.009766   
2026-05-31  79822.726562  98.487999  430.500000  84.800003  7444.250000   

               Nasdaq100        VIX  
Date                                 
2026-01-31  25552.390625  17.440001  
2026-02-28  24960.039062  19.860001  
2026-03-31  23740.189453  25.250000  
2026-04-30  27452.119141  16.889999  
2026-05-31  29366.939453  17.790001  


## 结果解读
### 数据含义
本步骤最终获取**真实可溯源**的原始数据：包含美联储宏观利率、货币、通胀指标，以及美股、长债、黄金、比特币、MCHI中国ETF、EEM新兴市场ETF等全品类资产月度数据，时间区间覆盖1993—2025年，数据来源权威、真实有效。

### 主要发现
1. 成功纳入 MCHI、EEM 新兴市场资产，为后续美联储政策外溢效应分析提供真实数据支撑；
2. 各资产时间维度完整连续，能够完整匹配多轮美联储加息、降息政策周期；
3. 比特币真实数据自2014年起有效，符合实际上市时间，样本区间客观真实。

### 局限性
1. 利用 yfinance获取数据过程中受网络超时限制，数据下载稳定性受限；
2. 比特币历史存续周期仅有约10年，覆盖货币政策轮次偏少，样本长度天然受限；